In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = '/home/mkeoliya/projects/CAREBench/data/ehrshot'

LABEL: LAB

In [ ]:
for task_name in ['anemia', 'hyperkalemia', 'hypoglycemia', 'hyponatremia', 'thrombocytopenia']:
    label_df = pd.read_csv(os.path.join(DATA_DIR, f'label_{task_name}.csv'))
    label_df = label_df.drop_duplicates()
    label_df = label_df.drop(columns=['label_type'])
    label_df = label_df.groupby(['patient_id', 'value']).min().reset_index().sort_values(['patient_id', 'value'])
    label_df.to_csv(os.path.join(DATA_DIR, f'label_{TASK}.csv'))

In [ ]:
for task_name in ['anemia', 'hyperkalemia', 'hypoglycemia', 'hyponatremia', 'thrombocytopenia']:
    label_df = pd.read_csv(os.path.join(DATA_DIR, f'label_{task_name}.csv'))

    three_df = label_df[label_df['value']==3]
    two_df = label_df[(label_df['value']==2) & (~label_df['patient_id'].isin(three_df['patient_id']))]
    one_df = label_df[(label_df['value']==1) & (~label_df['patient_id'].isin(three_df['patient_id'])) & (~label_df['patient_id'].isin(two_df['patient_id']))]
    neg_df = label_df[(label_df['value']==0) & (~label_df['patient_id'].isin(three_df['patient_id'])) 
                    & (~label_df['patient_id'].isin(two_df['patient_id'])) & (~label_df['patient_id'].isin(one_df['patient_id']))]
    
    print(len(three_df), len(two_df), len(one_df), len(neg_df))

    three_df = three_df.groupby('patient_id').min().reset_index()
    two_df = two_df.groupby('patient_id').min().reset_index()
    one_df = one_df.groupby('patient_id').min().reset_index()
    neg_df = neg_df.groupby('patient_id').min().reset_index()

    label_df = pd.concat([three_df, two_df, one_df, neg_df], ignore_index=True)
    label_df.to_csv(os.path.join(DATA_DIR, f'{TASK}_label.csv'))

LABEL: ICU STAY

In [ ]:
label_df = pd.read_csv(os.path.join(DATA_DIR, 'labeled_patients.csv'))
label_df = label_df.drop_duplicates()

In [ ]:
pos_df = label_df[label_df['value']==True]
pos_df = pos_df.groupby('patient_id').min().reset_index()

neg_df = label_df[(label_df['value']==False) & (~label_df['patient_id'].isin(pos_df['patient_id']))]
neg_df = neg_df.groupby('patient_id').min().reset_index()

In [ ]:
label_df = pd.concat([pos_df, neg_df], ignore_index=True)

In [ ]:
label_df.to_parquet(os.path.join(DATA_DIR, f'{TASK}_label.parquet'))

LABEL: CHEST X-RAY

In [ ]:
CHEXPERT_LABELS = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Lesion",
    "Lung Opacity",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

In [ ]:
def int_to_binary_label_list(label_int, length=14):
    binary_str = format(label_int, f'0{length}b')[::-1]
    return [int(bit) for bit in binary_str]

In [ ]:
label_df = pd.read_csv(os.path.join(DATA_DIR, 'labeled_patients.csv'))
label_df = label_df.drop_duplicates()

In [ ]:
values = label_df.groupby('value')['patient_id'].nunique().sort_values().reset_index()['value'].unique()
label_df['priority'] = label_df['value'].map({v: i for i, v in enumerate(values)})
label_df = label_df.sort_values(by=['patient_id', 'priority']).drop_duplicates('patient_id')
label_df = label_df.drop(columns='priority')

In [ ]:
label_df['value'] = label_df['value'].apply(int_to_binary_label_list)

In [ ]:
label_df.to_parquet(os.path.join(DATA_DIR, f'{TASK}_label.parquet'))

SAMPLE TIMES

In [ ]:
TASK = 'hypoglycemia'

In [ ]:
def assign_group_numbers(group):
    time_diff = group['start'].diff().fillna(pd.Timedelta(days=0))
    new_group = (time_diff >= pd.Timedelta(days=1)).cumsum()
    return new_group + 1

In [ ]:
label_df = pd.read_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}_label.parquet')

In [ ]:
df = pd.read_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}.parquet')
df['start'] = pd.to_datetime(df['start'])
df = df.sort_values(['patient_id', 'start'])
df['N'] = df.groupby('patient_id', group_keys=False).apply(assign_group_numbers).reset_index(drop=True)

In [ ]:
pred_df = df[['patient_id', 'start', 'N']].merge(label_df[['patient_id', 'prediction_time']], on='patient_id')
pred_df = pred_df[(pred_df['start'] - pred_df['prediction_time']).abs() < pd.Timedelta(hours=24)]
pred_df = pred_df.drop(columns=['start', 'prediction_time']).drop_duplicates()

In [ ]:
label_df = label_df.merge(pred_df.groupby('patient_id').min().reset_index(), on='patient_id')
label_df = label_df.rename(columns={'N': 'N_min'})
label_df = label_df.merge(pred_df.groupby('patient_id').max().reset_index(), on='patient_id')
label_df = label_df.rename(columns={'N': 'N_max'})

In [ ]:
df = df.merge(label_df[['patient_id', 'N_min', 'N_max']], on='patient_id')

In [ ]:
df.to_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}.parquet')

In [ ]:
label_df = label_df.drop(columns=['start', 'N_min', 'N_max'])

In [ ]:
start_df = df.groupby('patient_id')['start'].min().reset_index()
label_df = label_df.merge(start_df, on='patient_id', how='left')

In [ ]:
pos_df = label_df[label_df['value']]
pos_df = pos_df[(pos_df['prediction_time']  - pos_df['start']) >= pd.Timedelta(hours=1)]

neg_df = label_df[~label_df['value']]
neg_df = neg_df.drop(columns=['prediction_time'])

In [ ]:
delta = (pos_df['prediction_time'] - pos_df['start']).dt.total_seconds() / 3600
times = delta.sample(n=int(len(neg_df)*1.2), replace=True) + np.random.normal(0, 0.5, int(len(neg_df)*1.2))

In [ ]:
neg_df['Sample_time'] = [t for t in times if t > 1.5][:len(neg_df)]
neg_df['Sample_time'] = neg_df['Sample_time'].apply(lambda x: pd.Timedelta(hours=x)) + neg_df['start']

In [ ]:
label_df = pd.concat([pos_df, neg_df], ignore_index=True)

In [ ]:
label_df.to_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}_label.parquet')

DATASET

In [ ]:
def get_labs(phenotype):
    label_df = pd.read_csv(os.path.join(DATA_DIR, f'benchmark/label_{phenotype}.csv'))
    ids = label_df['patient_id'].unique()
    df = pd.read_parquet(os.path.join(DATA_DIR, f'ehrshot.parquet'))
    df = df[df['patient_id'].isin(ids)]
    return df

*ICU*

In [ ]:
df_icu = get_labs('guo_icu')
df_icu.to_parquet(os.path.join(DATA_DIR, 'benchmark/guo_icu/icu.parquet'))

*Chest Expert*

In [ ]:
df_chest = get_labs('chexpert')
df_chest.to_parquet(os.path.join(DATA_DIR, 'benchmark/chexpert/chexpert.parquet'))

*THROMBOCYTOPENIA*

In [ ]:
THROMBOCYTOPENIA = ['LOINC/LP393218-5', 'LOINC/LG32892-8', 'LOINC/777-3']

In [ ]:
df_thrombocytopenia = get_labs('lab_thrombocytopenia')
df_thrombocytopenia.to_parquet(os.path.join(DATA_DIR, 'thrombocytopenia.parquet'))

*HYPERGLYCEMIA*

In [ ]:
HYPERGLYCEMIA = ['SNOMED/33747003', 'LOINC/LP416145-3', 'LOINC/14749-6']

In [ ]:
df_hypoglycemia = get_labs('lab_hypoglycemia')
df_hypoglycemia.to_parquet(os.path.join(DATA_DIR, 'hypoglycemia.parquet'))

*HYPONATREMIA*

In [ ]:
HYPONATREMIA = ['LOINC/LG11363-5', 'LOINC/2951-2', 'LOINC/2947-0']

In [ ]:
df_hyponatremia = get_labs('lab_hyponatremia')
df_hyponatremia.to_parquet(os.path.join(DATA_DIR, 'hyponatremia.parquet'))

*ANEMIA*

In [ ]:
ANEMIA = ['LOINC/LP392452-1']

In [ ]:
df_anemia = get_labs('lab_anemia')
df_anemia.to_parquet(os.path.join(DATA_DIR, 'anemia.parquet'))

*HYPERKALEMIA*

In [ ]:
HYPERKALEMIA = ['LOINC/LG7931-1', 'LOINC/LP386618-5', 'LOINC/LG10990-6', 'LOINC/6298-4', 'LOINC/2823-3']

In [ ]:
df_hyperkalemia = get_labs('lab_hyperkalemia')
df_hyperkalemia.to_parquet(os.path.join(DATA_DIR, 'hyperkalemia.parquet'))